In [ ]:
#密度mapにグリッドを重ね書きするコード
#AMRの各レベルが適用されたブロックごとに色分けして表示（レベル０が基準）
#x-y平面用

import pyvista as pv
import numpy as np
import matplotlib.pyplot as plt
import os
from matplotlib.colors import LogNorm
from scipy.interpolate import griddata
import matplotlib.patches as patches
from matplotlib.gridspec import GridSpec
from collections import defaultdict
import matplotlib.patches as mpatches

# ============================================================
# 設定
# ============================================================
vtk_dir = os.path.expanduser("~/athena-project/results/〇〇")
output_dir = "./xy_density_maps_with_grid"
os.makedirs(output_dir, exist_ok=True)

# 計算領域の設定
Lbox = 44800.0
x_min, x_max = -Lbox/2, Lbox/2
y_min, y_max = -Lbox/2, Lbox/2

xc, yc = 0.0, 0.0
dpi = 200
figsize = (16, 8)

# AMR可視化の設定
SHOW_AMR_BLOCKS = True

# ベースグリッド設定（レベル0の解像度）
BASE_NX = 64  # レベル0での1辺あたりのセル数
BASE_DX = Lbox / BASE_NX  # レベル0のセルサイズ

# レベルごとの設定（レベル0からスタート）
# レベル0: ベース（最も粗い）
# レベル1: BASE_DX / 2
# レベル2: BASE_DX / 4
# レベル3: BASE_DX / 8
# レベル4: BASE_DX / 16
# レベル5: BASE_DX / 32（最も細かい）
LEVEL_COLORS = {
    0: 'gray',      # レベル0（ベース、最も粗い）
    1: 'blue',      # レベル1
    2: 'green',     # レベル2
    3: 'black',     # レベル3
    4: 'red',       # レベル4
    5: 'purple'     # レベル5（最も細かい）
}

def calculate_cell_size(level):
    """レベルからセルサイズを計算（レベルが上がるごとに半分になる）"""
    # レベル0: dx = BASE_DX
    # レベル1: dx = BASE_DX / 2
    # レベル2: dx = BASE_DX / 4
    # レベル3: dx = BASE_DX / 8
    # ...
    return BASE_DX / (2 ** level)

# ============================================================
# AMRブロック情報を抽出（ブロック単位）
# ============================================================
def extract_amr_block_info(grid, timestep, z_tolerance=5.0):
    """
    VTKグリッドからAMRブロック情報を抽出（ブロック単位）
    """
    cell_centers = grid.cell_centers().points
    
    if len(cell_centers) == 0:
        return None
    
    # z条件でフィルタリング（z=0付近のブロックのみ）
    mask_z = np.abs(cell_centers[:, 2]) <= z_tolerance
    if not np.any(mask_z):
        return None
    
    filtered_centers = cell_centers[mask_z]
    
    # セルサイズからレベルを計算
    x_coords = np.unique(np.sort(filtered_centers[:, 0]))
    if len(x_coords) > 1:
        dx_actual = np.min(np.diff(x_coords))
        # レベル = log2(ベースセルサイズ / 実際のセルサイズ)
        # 例: BASE_DX=7.0, dx_actual=3.5 → log2(2)=1 → level=1
        level = int(np.round(np.log2(BASE_DX / dx_actual)))
        level = max(0, min(level, 5))  # レベル0〜5
    else:
        dx_actual = BASE_DX
        level = 0
    
    # ブロックの境界を計算
    x_min_block = filtered_centers[:, 0].min() - dx_actual/2
    x_max_block = filtered_centers[:, 0].max() + dx_actual/2
    y_min_block = filtered_centers[:, 1].min() - dx_actual/2
    y_max_block = filtered_centers[:, 1].max() + dx_actual/2
    
    return {
        'bounds': (x_min_block, x_max_block, y_min_block, y_max_block),
        'level': level,
        'dx': dx_actual,
        'n_cells': len(filtered_centers)
    }

# ============================================================
# 重複する境界線を処理して描画する関数
# ============================================================
def draw_amr_blocks_without_duplicate_edges(ax, amr_blocks, alpha=0.7, linewidth=1.0):
    """
    隣接ブロックとの境界線が重ならないようにAMRブロックを描画
    各ブロックの右辺と上辺のみを描画し、左辺と下辺は描画しない
    """
    # ブロックをレベル順にソート（低レベルから描画）
    sorted_blocks = sorted(amr_blocks, key=lambda b: b['level'])
    
    for block in sorted_blocks:
        bounds = block['bounds']
        level = block['level']
        color = LEVEL_COLORS.get(level, 'white')
        
        x0, x1, y0, y1 = bounds[0], bounds[1], bounds[2], bounds[3]
        
        # 右辺（x = x1）
        ax.plot([x1, x1], [y0, y1], color=color, linewidth=linewidth, alpha=alpha)
        # 上辺（y = y1）
        ax.plot([x0, x1], [y1, y1], color=color, linewidth=linewidth, alpha=alpha)
        # 左辺と下辺は描画しない（隣接ブロックと重複を避ける）

# ============================================================
# VTK ファイル整理
# ============================================================
print("[INFO] Organizing VTK files...")
print(f"[INFO] Level 0 base grid: {BASE_NX}×{BASE_NX}×{BASE_NX} (dx = {BASE_DX:.3f})")
print(f"[INFO] AMR level cell sizes (each level doubles resolution):")
for level in range(0, 6):
    dx = calculate_cell_size(level)
    if level == 0:
        print(f"  Level {level} (base): dx = {dx:.4f}")
    else:
        print(f"  Level {level}: dx = {BASE_DX:.3f} / {2**level} = {dx:.4f}")

timestep_dict = defaultdict(list)
all_timesteps = set()

for f in os.listdir(vtk_dir):
    if f.endswith(".vtk") and f.startswith("Toyouchi.block"):
        try:
            if '.prim.' in f:
                timestep_str = f.split('.prim.out2.')[1].split('.')[0]
            else:
                timestep_str = f.split('.out2.')[1].split('.')[0]
            timestep = int(timestep_str)
            timestep_dict[timestep].append(os.path.join(vtk_dir, f))
            all_timesteps.add(timestep)
        except (IndexError, ValueError):
            continue

timesteps = sorted(all_timesteps)
if not timesteps:
    print("[ERROR] No timesteps found!")
    exit(1)

print(f"[INFO] Found {len(timesteps)} timesteps")
print(f"[INFO] Timestep range: {timesteps[0]} - {timesteps[-1]}")
print(f"[INFO] Files per timestep: {len(timestep_dict[timesteps[0]])}")

# ============================================================
# 密度変数名を検出
# ============================================================
print("[INFO] Detecting density variable name...")

test_file = timestep_dict[timesteps[0]][0]
test_grid = pv.read(test_file)
print(f"[INFO] Available arrays: {test_grid.array_names}")

density_name = None
for name in ['dens', 'density', 'rho', 'prim_dens', 'prim_density']:
    if name in test_grid.array_names:
        density_name = name
        break

if density_name is None:
    print(f"[ERROR] No density array found.")
    exit(1)

print(f"[INFO] Using density array: '{density_name}'")

# ============================================================
# 密度スケールの決定
# ============================================================
print("[INFO] Determining density scale...")

sample_indices = np.linspace(0, len(timesteps)-1, min(10, len(timesteps))).astype(int)
sample_timesteps = [timesteps[i] for i in sample_indices]

all_densities = []

for ts in sample_timesteps:
    for f in timestep_dict[ts]:
        try:
            grid = pv.read(f)
            all_densities.append(grid[density_name].flatten())
        except:
            continue

if all_densities:
    all_densities = np.concatenate(all_densities)
    positive_dens = all_densities[all_densities > 0]
    if len(positive_dens) > 0:
        vmin = np.percentile(positive_dens, 1)
        vmax = np.percentile(positive_dens, 99)
        print(f"[INFO] Density scale: {vmin:.3e} - {vmax:.3e}")
    else:
        vmin, vmax = 1e-6, 1e-2
else:
    vmin, vmax = 1e-6, 1e-2

# ============================================================
# メイン処理
# ============================================================
print("[INFO] Generating density maps with AMR block boundaries...")

grid_resolution = 800
inner_resolution = 600

for ts_idx, ts in enumerate(timesteps):
    print(f"[INFO] Processing timestep {ts} ({ts_idx+1}/{len(timesteps)})")
    
    points_list = []
    dens_list = []
    amr_blocks = []
    
    for f in timestep_dict[ts]:
        try:
            grid = pv.read(f)
            
            # 密度データ
            points = grid.cell_centers().points
            dens = grid[density_name]
            
            z_tolerance = BASE_DX
            mask_z = np.abs(points[:, 2]) <= z_tolerance
            if np.any(mask_z):
                points_list.append(points[mask_z])
                dens_list.append(dens[mask_z])
            
            # AMRブロック情報を抽出
            block_info = extract_amr_block_info(grid, ts, z_tolerance)
            if block_info is not None:
                amr_blocks.append(block_info)
                
        except Exception as e:
            continue
    
    if not points_list:
        print(f"[WARNING] No data for timestep {ts}, skipping...")
        continue
    
    pts_all = np.vstack(points_list)
    dens_all = np.hstack(dens_list)
    
    rho_max = np.nanmax(dens_all)
    rho_min = np.nanmin(dens_all)
    imax = np.nanargmax(dens_all)
    imin = np.nanargmin(dens_all)
    pos_max = pts_all[imax]
    pos_min = pts_all[imin]
    print(
        f"[INFO] step={ts:05d}  "
        f"rho_max={rho_max:.3e} at "
        f"({pos_max[0]:.1f},{pos_max[1]:.1f},{pos_max[2]:.1f})   "
        f"rho_min={rho_min:.3e} at "
        f"({pos_min[0]:.1f},{pos_min[1]:.1f},{pos_min[2]:.1f})"
    )
    
    # 半径を計算
    r_max = np.sqrt(pos_max[0]**2 +
                    pos_max[1]**2 +
                    pos_max[2]**2)
    r_min = np.sqrt(pos_min[0]**2 +
                    pos_min[1]**2 +
                    pos_min[2]**2)
    
    # ====================================================
    # 自動ズーム半径の決定
    # ====================================================

    rxy = np.sqrt(pts_all[:,0]**2 + pts_all[:,1]**2)
    
    # 原点に最も近いセル中心の半径
    rmin = np.min(rxy)

    # その20倍をズーム半径とする
    high_res_radius = 20.0 * rmin

    # 極端な値を防ぐ
    high_res_radius = np.clip(high_res_radius,
                              500.0,   # 最小500
                              5000.0)  # 最大5000

    print(f"[INFO] rmin = {rmin:.2f}")
    print(f"[INFO] zoom radius = {high_res_radius:.2f}")
    
    # 補間用グリッド
    xs_full = np.linspace(x_min, x_max, grid_resolution)
    ys_full = np.linspace(y_min, y_max, grid_resolution)
    X_full, Y_full = np.meshgrid(xs_full, ys_full)
    points_2d = np.column_stack([pts_all[:, 0], pts_all[:, 1]])
    dens_full = griddata(points_2d, dens_all, (X_full, Y_full), method='linear')
    
    # 拡大図用
    mask_inner = np.sqrt(pts_all[:, 0]**2 + pts_all[:, 1]**2) <= high_res_radius
    pts_inner = pts_all[mask_inner]
    dens_inner = dens_all[mask_inner]
    
    if len(pts_inner) > 10:
        xs_inner = np.linspace(-high_res_radius, high_res_radius, inner_resolution)
        ys_inner = np.linspace(-high_res_radius, high_res_radius, inner_resolution)
        X_inner, Y_inner = np.meshgrid(xs_inner, ys_inner)
        points_inner_2d = np.column_stack([pts_inner[:, 0], pts_inner[:, 1]])
        dens_inner_fine = griddata(points_inner_2d, dens_inner, (X_inner, Y_inner), method='nearest')
    else:
        X_inner, Y_inner, dens_inner_fine = None, None, None
    
    # ====================================================
    # プロット
    # ====================================================
    fig = plt.figure(figsize=figsize)
    gs = GridSpec(1, 4,width_ratios=[4, 4, 0.3, 1.3],wspace=0.4)
    
    # === 全体図 ===
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.set_aspect('equal')
    
    # 密度マップ
    im1 = ax1.pcolormesh(X_full, Y_full, dens_full,
                         cmap='inferno', norm=LogNorm(vmin=vmin, vmax=vmax),
                         shading='auto', rasterized=True)
    
    # AMRブロック境界を描画（重複を避ける）
    if SHOW_AMR_BLOCKS and amr_blocks:
        draw_amr_blocks_without_duplicate_edges(ax1, amr_blocks, alpha=0.8, linewidth=1.0)
    
    # 拡大領域表示
    rect = patches.Rectangle(
        (-high_res_radius, -high_res_radius),
        2*high_res_radius, 2*high_res_radius,
        linewidth=2, edgecolor='cyan', facecolor='none',
        linestyle='--', label=f'Zoom region (r={high_res_radius})'
    )
    ax1.add_patch(rect)
    ax1.plot(xc, yc, 'r+', markersize=12, markeredgewidth=2, label='Center')
    
    # 同心円
    for r in [50, 100, 150, 200]:
        if r <= Lbox/2:
            theta = np.linspace(0, 2*np.pi, 200)
            ax1.plot(xc + r*np.cos(theta), yc + r*np.sin(theta), 
                    'w--', alpha=0.3, linewidth=0.8)
    
    ax1.set_xlim(x_min, x_max)
    ax1.set_ylim(y_min, y_max)
    ax1.set_xlabel("X", fontsize=12)
    ax1.set_ylabel("Y", fontsize=12)
    ax1.set_title(f"Full Domain - AMR Blocks", fontsize=12, fontweight='bold')
    ax1.grid(True, alpha=0.2, linestyle='--', linewidth=0.5)
    
    # 凡例（レベル0〜5）
    legend_elements = []
    for level in range(0, 6):
        dx = calculate_cell_size(level)
        if level == 0:
            label = f'Level {level} (base, dx = {dx:.3f})'
        else:
            label = f'Level {level} (dx = {dx:.3f})'
        legend_elements.append(
            mpatches.Patch(facecolor='none', edgecolor=LEVEL_COLORS[level],
                         linewidth=1.5, label=label)
        )
    legend_elements.append(mpatches.Patch(facecolor='none', edgecolor='cyan', 
                                         linestyle='--', label='Zoom region'))
    ax1.legend(handles=legend_elements, loc='upper right', fontsize=7, framealpha=0.7)
    
    # === 拡大図 ===
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.set_aspect('equal')
    
    if X_inner is not None and dens_inner_fine is not None:
        im2 = ax2.pcolormesh(X_inner, Y_inner, dens_inner_fine,
                             cmap='inferno', norm=LogNorm(vmin=vmin, vmax=vmax),
                             shading='auto', rasterized=True)
        
        # 拡大図にもAMRブロック境界を表示
        if SHOW_AMR_BLOCKS and amr_blocks:
            # 拡大領域内のブロックをフィルタリング
            filtered_blocks = []
            for block in amr_blocks:
                bounds = block['bounds']
                if (bounds[0] <= high_res_radius and bounds[1] >= -high_res_radius and
                    bounds[2] <= high_res_radius and bounds[3] >= -high_res_radius):
                    # 表示範囲でクリップした境界を計算
                    x0 = max(bounds[0], -high_res_radius)
                    x1 = min(bounds[1], high_res_radius)
                    y0 = max(bounds[2], -high_res_radius)
                    y1 = min(bounds[3], high_res_radius)
                    
                    if x1 > x0 and y1 > y0:
                        filtered_blocks.append({
                            'bounds': (x0, x1, y0, y1),
                            'level': block['level']
                        })
            
            # 拡大図でも重複を避けて描画
            sorted_blocks = sorted(filtered_blocks, key=lambda b: b['level'])
            for block in sorted_blocks:
                bounds = block['bounds']
                level = block['level']
                color = LEVEL_COLORS.get(level, 'white')
                
                x0, x1, y0, y1 = bounds
                # 右辺と上辺のみ描画
                ax2.plot([x1, x1], [y0, y1], color=color, linewidth=1.2, alpha=0.8)
                ax2.plot([x0, x1], [y1, y1], color=color, linewidth=1.2, alpha=0.8)
        
        ax2.plot(xc, yc, 'r+', markersize=12, markeredgewidth=2, label='Center')
        
        # 同心円
        for r in [20, 40, 60, 80, 100]:
            theta = np.linspace(0, 2*np.pi, 200)
            ax2.plot(xc + r*np.cos(theta), yc + r*np.sin(theta), 
                    'w--', alpha=0.5, linewidth=0.8)
            
        ax2.set_xlim(-high_res_radius, high_res_radius)
        ax2.set_ylim(-high_res_radius, high_res_radius)
        ax2.set_xlabel("X", fontsize=12)
        ax2.set_ylabel("Y", fontsize=12)
        ax2.set_title(f"Zoomed Region (R={high_res_radius:.0f})")
        ax2.grid(True, alpha=0.2, linestyle='--', linewidth=0.5)
        ax2.legend(loc='upper right', fontsize=8, framealpha=0.7)
    else:
        ax2.text(0.5, 0.5, 'No data in inner region', 
                transform=ax2.transAxes, ha='center', va='center', fontsize=12)
        ax2.set_xlim(-high_res_radius, high_res_radius)
        ax2.set_ylim(-high_res_radius, high_res_radius)
    
    # カラーバー
    cax = fig.add_subplot(gs[0, 2])
    
    # rho_max, rho_min表示
    ax_info = fig.add_subplot(gs[0, 3])
    ax_info.axis('off')
    info_text = (
        f"rho_max\n"
        f"{rho_max:.3e}\n"
        f"(r = {r_max:.1f})\n\n"
        f"rho_min\n"
        f"{rho_min:.3e}\n"
        f"(r = {r_min:.1f})"
    )
    ax_info.text(
        0.05, 0.95,
        info_text,
        transform=ax_info.transAxes,
        fontsize=12,
        verticalalignment='top',
        family='monospace',
        bbox=dict(
            boxstyle='round',
            facecolor='whitesmoke',
            edgecolor='black',
            alpha=0.9
        )
    )
    
    from matplotlib.ticker import LogLocator, LogFormatterSciNotation
    from matplotlib.ticker import LogFormatterSciNotation
    cbar = fig.colorbar(im1, cax=cax, extend='both')
    ticks = np.logspace(np.log10(vmin),
                        np.log10(vmax),
                        8)
    cbar.set_ticks(ticks)
    cbar.ax.yaxis.set_major_formatter(
        LogFormatterSciNotation()
    )
    cbar.set_label("Density", fontsize=12)
    cbar.ax.tick_params(labelsize=10)
    
    # タイトル
    fig.suptitle(f"AMR Density Map with Block Boundaries\n"
                 f"Timestep: {ts:05d} | Variable: {density_name} | "
                 f"AMR Blocks: {len(amr_blocks)} | Base (Level 0) dx = {BASE_DX:.3f}",
                 fontsize=14, fontweight='bold', y=0.98)
    
    plt.subplots_adjust(top=0.92, bottom=0.08, left=0.05, right=0.95, wspace=0.25)
    
    # 保存
    png = os.path.join(output_dir, f"density_xy_timestep_{ts:05d}_amr_blocks.png")
    fig.savefig(png, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    
    if (ts_idx + 1) % 10 == 0:
        print(f"[INFO] Processed {ts_idx + 1}/{len(timesteps)} timesteps")

print(f"[INFO] All AMR density maps saved to {output_dir}")
print(f"[INFO] Total images: {len(timesteps)}")
print(f"\n[INFO] AMR Level Configuration (Level 0 = base):")
print(f"  Level 0 (base): {BASE_NX}×{BASE_NX} grid, dx = {BASE_DX:.4f}")
for level in range(1, 6):
    dx = calculate_cell_size(level)
    print(f"  Level {level}: dx = {BASE_DX:.4f} / {2**level} = {dx:.4f}")